# 07 — Functional-Preserving Evasion

**Masalah:** FGSM standar (T5) memperturbasi ke-9 fitur secara bebas di ruang ter-z-score — bisa menghasilkan flow yang **tak mungkin ada** (paket pecahan, durasi negatif, bytes < pkts). Reviewer Q1 menolak evasion semacam ini karena **tak dapat dikirim** di jaringan nyata.

**Solusi (kontribusi T8):** *Functional-Preserving Constraints* — perturbasi hanya diperbolehkan menghasilkan flow yang **valid secara protokol/fisik**. Perbandingan:
1. **Unconstrained evasion** (FGSM bebas, ala T5) — batas atas daya serang, tapi tak realistis.
2. **Functional-preserving evasion** (FGSM + proyeksi ke ruang valid) — realistis, benar-benar dapat dikirim.

**Pertanyaan inti:** seberapa besar penurunan MCC yang MASIH bisa dicapai penyerang ketika dibatasi ke perturbasi yang valid? Jika model tetap rentan → ancaman nyata; jika constraint memangkas daya serang → model lebih aman dari yang terlihat pada evaluasi tak-terbatas.

**Constraint per-fitur (Model A, 9 fitur), diterapkan di RUANG ASLI (un-scale):**
| Fitur | Constraint |
|---|---|
| duration | ≥ 0 |
| fwd_pkts, bwd_pkts | ≥ 0, integer |
| fwd_bytes, bwd_bytes | ≥ 0, dan ≥ pkts (tiap paket ≥ 1 byte) |
| fwd_mean, bwd_mean | ≥ 0, dikonsistenkan = bytes/pkts |
| src_load, dst_load | ≥ 0 |
| **arah serangan** | **monotonic**: penyerang hanya boleh MENAMBAH (padding/paket), tak boleh mengurangi trafik yang sudah terkirim |

**Model yang diserang:** (a) baseline single-source, (b) model few-shot-adapted (1% target, dari T7) — untuk melihat apakah adaptasi mengubah kerentanan.

> **Kejujuran:** perturbasi di ruang z-score; constraint fisik diterapkan setelah un-scale, lalu di-scale ulang. Semua angka apa adanya.

> Jalankan di SageMaker.

In [ ]:
# --- Bootstrap ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg}'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import pickle, os, json
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUT_JSON='../functional_preserving_evasion.json'
SEED=42; H=0.01; EPS_EVAL=[0.05,0.1,0.2]; MAXN=40000
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST))

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys())
# indeks fitur utk constraint
IX={c:i for i,c in enumerate(CANON)}

def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan)
    out=out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

def make_xgb():
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=SEED,tree_method='hist')

def ev(yt,yp):
    return dict(mcc=float(matthews_corrcoef(yt,yp)),f1=float(f1_score(yt,yp,zero_division=0)),
                acc=float(accuracy_score(yt,yp)),confusion=confusion_matrix(yt,yp).tolist())

In [ ]:
# --- Muat data + z-score per dataset ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values

Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC tr/te:',Xc_tr.shape,Xc_te.shape,'| UNSW tr/te:',Xu_tr.shape,Xu_te.shape)

In [ ]:
# --- Serangan: finite-diff saliency + FGSM (biner) ---
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))

def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S

def fgsm_unconstrained(X,S,eps):
    return X + eps*np.sign(S)

def project_functional(X_adv_scaled, scaler):
    """Proyeksikan ke ruang valid: un-scale -> terapkan constraint fisik -> re-scale."""
    Xo = X_adv_scaled * scaler.scale_ + scaler.mean_   # ke ruang asli
    Xo = Xo.copy()
    # non-negatif utk semua fitur volume/laju/durasi
    Xo = np.clip(Xo, 0.0, None)
    # paket integer
    Xo[:, IX['fwd_pkts']] = np.round(Xo[:, IX['fwd_pkts']])
    Xo[:, IX['bwd_pkts']] = np.round(Xo[:, IX['bwd_pkts']])
    # bytes >= pkts (tiap paket minimal 1 byte)
    Xo[:, IX['fwd_bytes']] = np.maximum(Xo[:, IX['fwd_bytes']], Xo[:, IX['fwd_pkts']])
    Xo[:, IX['bwd_bytes']] = np.maximum(Xo[:, IX['bwd_bytes']], Xo[:, IX['bwd_pkts']])
    # mean = bytes/pkts (konsisten; hindari kontradiksi)
    with np.errstate(divide='ignore', invalid='ignore'):
        fm = np.where(Xo[:,IX['fwd_pkts']]>0, Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']], 0.0)
        bm = np.where(Xo[:,IX['bwd_pkts']]>0, Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']], 0.0)
    Xo[:, IX['fwd_mean']] = fm
    Xo[:, IX['bwd_mean']] = bm
    # kembali ke ruang z-score
    return (Xo - scaler.mean_) / scaler.scale_

def fgsm_functional(X_scaled, S, eps, X_orig_scaled, scaler):
    """FGSM + constraint monotonic (hanya boleh menambah) + proyeksi fungsional."""
    X_adv = X_scaled + eps*np.sign(S)
    # monotonic: penyerang hanya boleh MENAMBAH trafik (di ruang asli)
    Xo_adv = X_adv * scaler.scale_ + scaler.mean_
    Xo_ref = X_orig_scaled * scaler.scale_ + scaler.mean_
    add_only = [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]
    for j in add_only:
        Xo_adv[:, j] = np.maximum(Xo_adv[:, j], Xo_ref[:, j])  # tak boleh di bawah nilai asli
    X_adv = (Xo_adv - scaler.mean_) / scaler.scale_
    return project_functional(X_adv, scaler)

In [ ]:
# --- Fungsi evaluasi serangan pada satu model+data ---
def attack_eval(model, X, y, scaler, tag):
    rng=np.random.RandomState(SEED)
    n=min(MAXN,len(X)); idx=rng.choice(len(X),n,replace=False)
    Xs, ys = X[idx], y[idx]
    S=saliency(model,Xs,ys)
    out={'clean': ev(ys, model.predict(Xs))}
    for e in EPS_EVAL:
        Xu = fgsm_unconstrained(Xs,S,e)
        Xf = fgsm_functional(Xs,S,e,Xs,scaler)
        out[f'unconstrained_eps{e}'] = ev(ys, model.predict(Xu))
        out[f'functional_eps{e}']    = ev(ys, model.predict(Xf))
    print(f'[{tag}] clean MCC={out["clean"]["mcc"]:.4f}')
    for e in EPS_EVAL:
        u=out[f'unconstrained_eps{e}']['mcc']; f=out[f'functional_eps{e}']['mcc']
        print(f'   eps={e}: unconstrained MCC={u:+.4f} | functional MCC={f:+.4f}  (selisih daya serang={f-u:+.4f})')
    return out

In [ ]:
# --- Latih model yang akan diserang ---
# (a) baseline single-source: CIC-only & UNSW-only
m_cic = make_xgb(); m_cic.fit(Xc_tr, yc_tr)
m_unsw = make_xgb(); m_unsw.fit(Xu_tr, y_utr)

# (b) few-shot adapted (1% target), meniru T7
rng=np.random.RandomState(SEED)
# CIC + 1% UNSW
n1=int(len(Xu_tr)*0.01); i1=rng.choice(len(Xu_tr),n1,replace=False)
m_cic_adapt=make_xgb(); m_cic_adapt.fit(np.vstack([Xc_tr,Xu_tr[i1]]), np.concatenate([yc_tr,y_utr[i1]]))
# UNSW + 1% CIC
n2=int(len(Xc_tr)*0.01); i2=rng.choice(len(Xc_tr),n2,replace=False)
m_unsw_adapt=make_xgb(); m_unsw_adapt.fit(np.vstack([Xu_tr,Xc_tr[i2]]), np.concatenate([y_utr,yc_tr[i2]]))
print('4 model dilatih: baseline CIC/UNSW + adapted CIC/UNSW')

In [ ]:
# --- Jalankan serangan: baseline vs adapted, in-domain test ---
results={}
print('='*70); print('SERANGAN pada CIC test (model dilatih CIC)'); print('='*70)
results['cic_baseline'] = attack_eval(m_cic, Xc_te, yc_te, scc, 'CIC baseline')
results['cic_adapted']  = attack_eval(m_cic_adapt, Xc_te, yc_te, scc, 'CIC adapted(+1% UNSW)')
print('\n'+'='*70); print('SERANGAN pada UNSW test (model dilatih UNSW)'); print('='*70)
results['unsw_baseline'] = attack_eval(m_unsw, Xu_te, y_ute, scu, 'UNSW baseline')
results['unsw_adapted']  = attack_eval(m_unsw_adapt, Xu_te, y_ute, scu, 'UNSW adapted(+1% CIC)')

In [ ]:
# --- Ringkasan: daya serang unconstrained vs functional (eps=0.1) ---
print('RINGKASAN (eps=0.1) — MCC di bawah serangan:')
print(f"{'model':<26}{'clean':>8}{'unconstr':>10}{'functional':>12}")
rows=[]
for k,v in results.items():
    c=v['clean']['mcc']; u=v['unconstrained_eps0.1']['mcc']; f=v['functional_eps0.1']['mcc']
    print(f"{k:<26}{c:>8.4f}{u:>10.4f}{f:>12.4f}")
    rows.append(dict(model=k, clean=round(c,4), unconstrained=round(u,4), functional=round(f,4)))

print('\nInterpretasi: functional MCC biasanya > unconstrained MCC (constraint memangkas daya serang).')
print('Selisih (functional - unconstrained) = seberapa banyak serangan "tak realistis" melebih-lebihkan ancaman.')

meta=dict(
  deskripsi='Functional-preserving evasion vs unconstrained FGSM (Model A biner). Constraint di ruang asli.',
  features=CANON, eps_eval=EPS_EVAL,
  constraints='non-neg; pkts integer; bytes>=pkts; mean=bytes/pkts; monotonic add-only pd pkts/bytes/duration',
  summary_eps01=rows, results=results,
)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print('\nSaved:', OUT_JSON)